# Streaming and Live Data Tutorial

This tutorial demonstrates QuantStrata's streaming and live trading infrastructure: replay stream, paper brokerage adapter, and StreamingEngine with the same strategy signature as backtesting.

**Topics covered:**
- Streaming protocol (async stream of timestamp, Market)
- ReplayStreamProvider from a list of (timestamp, Market)
- PaperBrokerageAdapter and apply_market for simulated fills
- StreamingEngine: strategy(market, portfolio, context) -> orders
- Same strategy usable in backtest and streaming

In [ ]:
import sys
sys.path.insert(0, '../../..')

import asyncio
import numpy as np

print("Setup complete.")

## 1. Minimal market snapshots and replay stream

Build a few (timestamp, Market) snapshots and a ReplayStreamProvider. No external API.

In [ ]:
from src.marketdata.core.market import Market
from src.marketdata.core.interfaces import Quote
from src.marketdata.core.ids import MarketId
from src.marketdata.providers.streaming import ReplayStreamProvider

spot_id = MarketId("FX", "SPOT", "EURUSD")

# Three snapshots: price 1.08, 1.10, 1.12
markets = [
    Market(asof="2024-01-01", quotes={spot_id: Quote(value=1.08)}, curves={}, vols={}),
    Market(asof="2024-01-02", quotes={spot_id: Quote(value=1.10)}, curves={}, vols={}),
    Market(asof="2024-01-03", quotes={spot_id: Quote(value=1.12)}, curves={}, vols={}),
]
timestamps = ["2024-01-01", "2024-01-02", "2024-01-03"]
snapshots = list(zip(timestamps, markets))

stream_provider = ReplayStreamProvider(snapshots=snapshots)
print("ReplayStreamProvider created with", len(snapshots), "snapshots.")

## 2. Paper adapter and strategy

Paper adapter simulates order execution. Strategy returns orders (objects with instrument_id, quantity); same signature as backtesting.

In [ ]:
from src.streaming import PaperBrokerageAdapter, StreamingEngine
from src.marketdata.core.ids import MarketId

class SimpleOrder:
    def __init__(self, instrument_id: str, quantity: float):
        self.instrument_id = instrument_id
        self.quantity = quantity

def get_price(inst_id: str, market):
    # Map instrument_id (e.g. "FX.SPOT.EURUSD") to MarketId and return quote
    mid = MarketId.parse(inst_id)
    return float(market.quote(mid))

def my_strategy(market, portfolio, context):
    orders = []
    if context.step == 0:
        orders.append(SimpleOrder(instrument_id=spot_id.key(), quantity=10_000.0))
    return orders

adapter = PaperBrokerageAdapter(initial_cash=100_000.0)
engine = StreamingEngine()
print("Strategy and adapter ready.")

## 3. Run streaming engine

Run the strategy on the replay stream; paper adapter fills orders at get_price(instrument_id, market).

In [ ]:
async def run():
    result = await engine.run_async(
        stream_provider=stream_provider,
        brokerage_adapter=adapter,
        strategy=my_strategy,
        get_price=get_price,
    )
    return result

result = asyncio.run(run())

print("Steps processed:", result.steps_processed)
print("Orders submitted:", len(result.orders_submitted))
positions = adapter.get_positions()
print("Positions:", [(p.instrument_id, p.quantity, p.avg_price) for p in positions])
print("Cash:", adapter.get_cash())